# Section 3 — VLM logit lens (and Workbench)

The previous two sections show how cheap it is to add interpretability
ops to existing PyTorch pipelines with `nnsight` + NDIF. But each
research-quality visualisation still wants its own UI: click-through
patches, layer sliders, color legends, the lot. Re-implementing those
per paper is exactly the duplicated-effort the talk argues against.

**Workbench** ([github.com/ndif-team/workbench](https://github.com/ndif-team/workbench))
is an interpretability UI built directly on `nnsight` + NDIF. Each
"tool" is a thin backend route that runs the trace on NDIF plus a
React widget that renders the result. Adding a tool is on the order
of: write the trace in nnsight, define the result schema, drop in a
visualization component. The talk includes a live demo of adding the
**VLM Logit Lens** tool — a per-layer next-token decoder over the
LLaVA image-patch positions — to the workbench in under an hour.

The rest of this section reproduces the lens technique inline so you
can run it on Colab without leaving the notebook — same model
(LLaVA-1.5-7b-hf), same trace shape, just rendered with matplotlib
instead of the polished React widget.

## The technique - LLaVA - Large Language And Vision Assistan
LLaVA-1.5 is a decoder-only Llama backbone fed image-patch tokens
(produced by a CLIP vision tower + small projector) interleaved with
the text prompt. The **logit lens** trick: at each of the 32 decoder
layers, apply the model's final RMSNorm + unembedding (`lm_head`) to
the residual stream as if it were the last layer, and read off the
top-1 token at every position.

For the image-token positions specifically, this becomes a 24 × 24
grid (576 patches) of "what would the model say each patch is, if we
stopped at layer L?". Plot it and you get a crude but interpretable
per-patch semantic map per layer.

The whole thing is a single `model.trace(prompt, images=[image])`
call: capture each layer's `output`, feed it through `norm` and
`lm_head` inside the same trace (calling wrapped modules as functions
inside a trace dispatches to their `forward()` and bypasses nnsight's
interleaving hooks — i.e. it's just the linear math we want), and
save the top-1 token id per (layer, position).

Original work: [Towards Interpreting Visual Information Processing in
Vision-Language Models, Neo et al. 2024](https://arxiv.org/abs/2410.07149).

![llava_architecture](llava_architecture.jpg)

In [ ]:
import os
import torch
from huggingface_hub import snapshot_download
from nnsight import VisionLanguageModel

local_model_path = "models/llava-1.5-7b-hf"
 
# Strict offline locks to block all remote pings
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# Load the local model into nnsight with local dispatching
llava = VisionLanguageModel(
    local_model_path, 
    dispatch=True, 
    torch_dtype=torch.float16, 
    device_map="auto"
)


## Load the demo image

Reproducible URL pointing at the same image we use throughout the
talk's accompanying code (`3_VLM_Lens/images/img.jpg` in this repo).

In [ ]:
import PIL.Image
PROMPT_3 = "USER: <image>\nDescribe the image. ASSISTANT:"

image_3 = PIL.Image.open("./img.jpg").convert("RGB")
display(image_3.resize((256, 256)))

## Trace and capture top-1 tokens per (layer, position)

We capture just the argmax (the top-1 token id) at each layer × each
position rather than the full top-k probabilities — keeps the wire
payload from NDIF small (~32 × seq_len ints instead of ~32 × seq_len ×
32064 floats).

In [ ]:
# GPU:

with llava.trace(PROMPT_3, images=[image_3], remote=False) as tracer:
    top1_per_layer = list().save()
    for layer in llava.model.language_model.layers:
        # `layer.output` is the residual stream after this block.
        # `model.lm_head(model.model.language_model.norm(...))` is the same
        # final readout the model itself does at the end.
        logits = llava.lm_head(llava.model.language_model.norm(layer.output))
        top1_per_layer.append(logits.argmax(dim=-1))  # [B, seq]

print(f"captured {len(top1_per_layer)} layers")
print(f"per-layer top1 shape: {tuple(top1_per_layer[0].shape)}")

## Build per-position labels

The processor expands the single `<image>` placeholder into 576 image
tokens. We mirror that expansion so each row of the lens table is
labelled either as a text token or as `<IMGxxx>` for one of the 576
patches.

In [ ]:
import util
tokenizer = llava.tokenizer
position_labels = util.build_position_labels(tokenizer, PROMPT_3)

assert (
    len(position_labels) == top1_per_layer[0].shape[1]
), f"label count {len(position_labels)} != seq len {top1_per_layer[0].shape[1]}"
print(
    f"sequence length: {len(position_labels)} "
    f"(== {util.LLAVA_IMAGE_GRID ** 2} image tokens + "
    f"{len(position_labels) - util.LLAVA_IMAGE_GRID ** 2} text tokens)"
)

## Compact lens table — text tokens only

Show what the model "decides" at each text-token position across a
sample of layers. The last-position prediction is the actual next
token the model would emit; earlier positions show what's at each
text token's slot.

In [ ]:
print("layers:", len(top1_per_layer))
print("first type:", type(top1_per_layer[0]) if top1_per_layer else None)
print("first shape:", tuple(top1_per_layer[0].shape) if top1_per_layer else None)
print("labels:", len(position_labels))
assert len(top1_per_layer) > 0
assert len(position_labels) == top1_per_layer[0].shape[1]

In [ ]:
util.show_lens_table(position_labels, top1_per_layer, tokenizer)

## Per-patch segmentation at a chosen layer

Color the 24 × 24 grid by the top-1 token at each patch — same colour
for the same predicted token. Overlay on the image so you can see
which patches the model lumps together semantically at this layer.
Edit `SEG_LAYER` to scroll through the depth of the model.

In [ ]:
SEG_LAYER = 22  # Works best for this prompt

util.show_patch_segmentation(
    image=image_3,
    position_labels=position_labels,
    top1_per_layer=top1_per_layer,
    tokenizer=tokenizer,
    layer=SEG_LAYER,
)

## Where to go from here

What we just did inline — one trace, capture per-layer top-1s, render
two views — is exactly the data path the full Workbench VLM Lens
tool runs. The Workbench instance adds:

  * the full top-5 (probabilities) per cell, surfaced via tooltip
  * a layer slider and a min-probability threshold slider for the
    segmentation widget
  * a click-to-recolour swatch per token in the legend
  * red bounding-outlines around all blobs of a hovered legend entry
  * click-to-lock-patch + auto-scroll the table to the locked row

…all of which is just a React widget on top of the same NDIF-served
trace. If you build a new vision interp method, that's the
template — wire the trace in `nnsight`, ship the result through
NDIF, render with whatever UI fits your audience.